# Modelagem e avaliação de previsões

## Objetivo

Nesta etapa, transformaremos os achados da análise exploratória em previsões semanais de internações por VSR.

O objetivo não será apenas encontrar o menor erro médio. Os modelos serão comparados de forma justa, usando os mesmos cortes temporais, os mesmos horizontes de previsão e métricas que mantenham interpretação em número de casos.

A análise começará por Brasil e regiões, que possuem séries mais densas e contínuas. Em seguida, as Unidades da Federação serão incorporadas de forma estruturada, considerando seu volume de casos, intermitência e proporção de semanas sem registros.

Como os dados possuem uma hierarquia geográfica — UF, região e Brasil —, também avaliaremos se as previsões são coerentes entre esses níveis. Por exemplo, a previsão para o Brasil deve ser compatível com a soma das previsões regionais e estaduais.

## Etapas previstas

1. Carregar a base madura e o calendário de validação temporal.
2. Formalizar a estrutura hierárquica formada por 27 UFs, 5 regiões e Brasil.
3. Construir referências simples de previsão, incluindo as referências ingênua e ingênua sazonal.
4. Calcular métricas de erro por série, modelo e horizonte.
5. Ajustar modelos que representem nível, tendência, dependência temporal e sazonalidade.
6. Verificar diagnósticos, limitações e estabilidade dos modelos entre períodos.
7. Produzir previsões iniciais para os diferentes níveis geográficos.
8. Aplicar e comparar métodos de reconciliação hierárquica, como bottom-up, top-down e MinT.
9. Comparar resultados nos horizontes de 1, 4, 8 e 13 semanas.
10. Registrar decisões, resultados e limitações para a entrega final.

## Resultado esperado

Ao final, teremos uma comparação reproduzível entre referências simples, modelos de séries temporais e previsões hierarquicamente reconciliadas.

A escolha final considerará desempenho preditivo, estabilidade entre períodos, interpretação epidemiológica, coerência entre UF, região e Brasil e utilidade para antecipar demanda por atendimento hospitalar.

## Etapa 20 — Preparação das entradas para previsão

Nesta etapa, carregaremos a base semanal madura e o calendário de validação temporal que serão utilizados em toda a modelagem.

O objetivo é confirmar que as séries de Brasil e regiões possuem as colunas necessárias, não apresentam duplicidades, usam apenas semanas maduras e correspondem exatamente aos valores observados registrados no calendário de validação. Esta verificação garante que todos os modelos posteriores sejam comparados sobre a mesma informação de entrada.

In [ ]:
# ETAPA 20 — Preparação das entradas para a modelagem
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# Caminhos fixos do ambiente atual no Google Colab
# ============================================================

BASE_DIR = Path("/content/tratamento_vsr")

ARQUIVO_PAINEL = (
    BASE_DIR
    / "dados_tratados"
    / "painel_vsr_semanal_com_maturidade.parquet"
)

ARQUIVO_CALENDARIO = (
    BASE_DIR
    / "saidas"
    / "etapa_19_validacao_temporal"
    / "calendario_validacao_temporal.csv"
)

DIRETORIO_RESULTADOS = (
    BASE_DIR
    / "saidas"
    / "etapa_20_preparacao_modelagem"
)

DIRETORIO_RESULTADOS.mkdir(parents=True, exist_ok=True)

if not ARQUIVO_PAINEL.exists():
    raise FileNotFoundError(f"Painel não encontrado: {ARQUIVO_PAINEL}")

if not ARQUIVO_CALENDARIO.exists():
    raise FileNotFoundError(
        f"Calendário não encontrado: {ARQUIVO_CALENDARIO}"
    )

print("Painel semanal:")
print(ARQUIVO_PAINEL)

print("\nCalendário de validação:")
print(ARQUIVO_CALENDARIO)


# ============================================================
# Leitura dos arquivos
# ============================================================

painel_original = pd.read_parquet(ARQUIVO_PAINEL)
calendario = pd.read_csv(ARQUIVO_CALENDARIO)

print(f"\nLinhas do painel original: {len(painel_original):,}")
print(f"Linhas do calendário: {len(calendario):,}")


# ============================================================
# Identificação explícita das colunas necessárias
# ============================================================

def localizar_coluna(df, candidatas, nome_logico):
    mapa = {
        coluna.strip().lower(): coluna
        for coluna in df.columns
    }

    for candidata in candidatas:
        if candidata.lower() in mapa:
            return mapa[candidata.lower()]

    raise KeyError(
        f"Não foi possível localizar a coluna de '{nome_logico}'.\n"
        f"Candidatas procuradas: {candidatas}\n"
        f"Colunas disponíveis: {list(df.columns)}"
    )


COLUNA_DATA = localizar_coluna(
    painel_original,
    [
        "semana",
        "semana_inicio",
        "semana_epi_inicio",
        "data_inicio_semana",
        "data_semana",
        "data_referencia",
    ],
    "data semanal",
)

COLUNA_CASOS = localizar_coluna(
    painel_original,
    [
        "casos_vsr",
        "casos_vsr_observados",
        "casos",
        "quantidade_casos",
    ],
    "quantidade de casos de VSR",
)

COLUNA_MATURA = localizar_coluna(
    painel_original,
    [
        "semana_madura_13s",
        "semana_madura",
        "semana_eh_madura",
        "e_madura",
        "eh_madura",
        "matura",
    ],
    "indicador de semana madura",
)

COLUNA_NIVEL = localizar_coluna(
    painel_original,
    ["nivel"],
    "nível geográfico",
)

COLUNA_UNIDADE = localizar_coluna(
    painel_original,
    ["unidade", "uf", "sigla_uf"],
    "unidade geográfica",
)

COLUNA_REGIAO = localizar_coluna(
    painel_original,
    ["regiao"],
    "região geográfica",
)

mapeamento_colunas = pd.DataFrame(
    {
        "campo_logico": [
            "data",
            "casos_vsr",
            "semana_madura",
            "nivel",
            "unidade",
            "regiao",
        ],
        "coluna_original": [
            COLUNA_DATA,
            COLUNA_CASOS,
            COLUNA_MATURA,
            COLUNA_NIVEL,
            COLUNA_UNIDADE,
            COLUNA_REGIAO,
        ],
    }
)

print("\nMapeamento de colunas:")
display(mapeamento_colunas)


# ============================================================
# Padronização da base de entrada
# ============================================================

def converter_para_booleano(serie):
    if pd.api.types.is_bool_dtype(serie):
        return serie.fillna(False)

    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce").fillna(0).eq(1)

    valores_texto = serie.astype("string").str.strip().str.upper()

    return valores_texto.isin(
        ["1", "TRUE", "T", "SIM", "S", "YES", "Y"]
    )


painel = painel_original.rename(
    columns={
        COLUNA_DATA: "data",
        COLUNA_CASOS: "casos_vsr",
        COLUNA_MATURA: "semana_madura",
        COLUNA_NIVEL: "nivel",
        COLUNA_UNIDADE: "unidade",
        COLUNA_REGIAO: "regiao",
    }
).copy()

painel = painel[
    [
        "nivel",
        "unidade",
        "regiao",
        "data",
        "casos_vsr",
        "semana_madura",
    ]
].copy()

painel["data"] = pd.to_datetime(painel["data"], errors="coerce")
painel["casos_vsr"] = pd.to_numeric(
    painel["casos_vsr"],
    errors="coerce",
)
painel["semana_madura"] = converter_para_booleano(
    painel["semana_madura"]
)

painel["nivel"] = (
    painel["nivel"].astype("string").str.strip().str.upper()
)

painel["unidade"] = (
    painel["unidade"].astype("string").str.strip().str.upper()
)

painel["regiao"] = (
    painel["regiao"].astype("string").str.strip().str.upper()
)

if painel["data"].isna().any():
    raise ValueError("Existem datas inválidas no painel semanal.")


# Diagnóstico: os valores ausentes são registrados antes do recorte.
# Eles não serão alterados nem preenchidos automaticamente.
diagnostico_ausentes = (
    painel
    .assign(casos_vsr_ausente=painel["casos_vsr"].isna())
    .groupby(["nivel", "semana_madura"], dropna=False)
    .agg(
        linhas=("casos_vsr", "size"),
        casos_vsr_ausentes=("casos_vsr_ausente", "sum"),
    )
    .reset_index()
)

diagnostico_ausentes.to_csv(
    DIRETORIO_RESULTADOS
    / "diagnostico_ausentes_casos_entrada_modelagem.csv",
    index=False,
)


# ============================================================
# Recorte inicial: Brasil e regiões, apenas semanas maduras
# ============================================================

painel_modelagem = painel.loc[
    painel["nivel"].isin(["BRASIL", "REGIAO"])
    & painel["semana_madura"]
].copy()

painel_modelagem = painel_modelagem.sort_values(
    ["nivel", "unidade", "data"]
).reset_index(drop=True)

if painel_modelagem.empty:
    raise ValueError(
        "O painel de Brasil e regiões ficou vazio após o filtro."
    )

if painel_modelagem["casos_vsr"].isna().any():
    exemplos = (
        painel_modelagem.loc[
            painel_modelagem["casos_vsr"].isna(),
            ["nivel", "unidade", "regiao", "data", "semana_madura"],
        ]
        .head(20)
    )

    display(exemplos)

    raise ValueError(
        "Existem valores ausentes de casos_vsr dentro das séries maduras "
        "de Brasil e regiões. A modelagem não pode continuar antes "
        "de investigar esses registros."
    )

if (painel_modelagem["casos_vsr"] < 0).any():
    raise ValueError(
        "Foram encontrados números negativos de casos na base de modelagem."
    )

if painel_modelagem.duplicated(
    ["nivel", "unidade", "data"]
).any():
    raise ValueError(
        "Existem duplicidades em nível + unidade + data "
        "na base de modelagem."
    )

resumo_entrada = (
    painel_modelagem
    .groupby(["nivel", "unidade", "regiao"], dropna=False)
    .agg(
        primeira_semana=("data", "min"),
        ultima_semana=("data", "max"),
        semanas=("data", "nunique"),
        casos_total=("casos_vsr", "sum"),
        media_semanal=("casos_vsr", "mean"),
        semanas_zero=(
            "casos_vsr",
            lambda serie: int((serie == 0).sum()),
        ),
    )
    .reset_index()
)

resumo_entrada["percentual_semanas_zero"] = (
    100
    * resumo_entrada["semanas_zero"]
    / resumo_entrada["semanas"]
)


# ============================================================
# Validação contra o calendário temporal definido na Etapa 19
# ============================================================

calendario["data_origem"] = pd.to_datetime(
    calendario["data_origem"],
    errors="coerce",
)

calendario["data_alvo"] = pd.to_datetime(
    calendario["data_alvo"],
    errors="coerce",
)

if calendario[["data_origem", "data_alvo"]].isna().any().any():
    raise ValueError(
        "Existem datas inválidas no calendário de validação."
    )

calendario["nivel"] = (
    calendario["nivel"].astype("string").str.strip().str.upper()
)

calendario["unidade"] = (
    calendario["unidade"].astype("string").str.strip().str.upper()
)

calendario_validacao = calendario.merge(
    painel_modelagem[
        [
            "nivel",
            "unidade",
            "data",
            "casos_vsr",
            "semana_madura",
        ]
    ],
    left_on=["nivel", "unidade", "data_alvo"],
    right_on=["nivel", "unidade", "data"],
    how="left",
    validate="many_to_one",
)

calendario_validacao["alvo_encontrado"] = (
    calendario_validacao["casos_vsr"].notna()
)

calendario_validacao["alvo_maduro"] = (
    calendario_validacao["semana_madura"].fillna(False)
)

calendario_validacao["valor_confere"] = np.isclose(
    calendario_validacao["casos_observados_alvo"],
    calendario_validacao["casos_vsr"],
    equal_nan=False,
)

calendario_validacao["diferenca_casos"] = (
    calendario_validacao["casos_observados_alvo"]
    - calendario_validacao["casos_vsr"]
)

if not calendario_validacao["alvo_encontrado"].all():
    raise ValueError(
        "Há alvos do calendário ausentes no painel de modelagem."
    )

if not calendario_validacao["alvo_maduro"].all():
    raise ValueError(
        "Há alvos do calendário que não são semanas maduras."
    )

if not calendario_validacao["valor_confere"].all():
    raise ValueError(
        "Os casos observados do calendário não conferem com o painel."
    )


# ============================================================
# Registro da qualidade da entrada e salvamento das saídas
# ============================================================

perfil_campos = pd.DataFrame(
    {
        "campo": painel_modelagem.columns,
        "tipo": painel_modelagem.dtypes.astype(str).values,
        "ausentes": painel_modelagem.isna().sum().values,
        "percentual_ausentes": (
            100 * painel_modelagem.isna().mean().values
        ).round(4),
    }
)

painel_modelagem.to_parquet(
    DIRETORIO_RESULTADOS
    / "painel_modelagem_brasil_regioes.parquet",
    index=False,
)

calendario.to_csv(
    DIRETORIO_RESULTADOS
    / "calendario_validacao_modelagem.csv",
    index=False,
)

mapeamento_colunas.to_csv(
    DIRETORIO_RESULTADOS
    / "mapeamento_colunas_entrada_modelagem.csv",
    index=False,
)

resumo_entrada.to_csv(
    DIRETORIO_RESULTADOS
    / "resumo_entrada_modelagem.csv",
    index=False,
)

perfil_campos.to_csv(
    DIRETORIO_RESULTADOS
    / "perfil_campos_entrada_modelagem.csv",
    index=False,
)

calendario_validacao.to_csv(
    DIRETORIO_RESULTADOS
    / "validacao_calendario_entrada_modelagem.csv",
    index=False,
)

print("\nEtapa 20 concluída com sucesso.")
print(
    f"Linhas maduras para modelagem: "
    f"{len(painel_modelagem):,}"
)
print(
    "Séries disponíveis: "
    f"{painel_modelagem[['nivel', 'unidade']].drop_duplicates().shape[0]}"
)
print(
    f"Linhas do calendário validadas: "
    f"{len(calendario_validacao):,}"
)
print(f"\nSaídas salvas em:\n{DIRETORIO_RESULTADOS}")

## Etapa 21 — Referências simples de previsão

Nesta etapa, construiremos previsões de referência para Brasil e regiões antes de ajustar modelos mais complexos.

O objetivo é estabelecer um padrão mínimo de desempenho. Serão utilizadas uma referência ingênua, que mantém o último valor observado, e uma referência ingênua sazonal, que utiliza a mesma semana epidemiológica da temporada anterior.

In [ ]:
# ETAPA 21 — Referências simples de previsão

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path("/content/tratamento_vsr")

ARQUIVO_PAINEL_ORIGINAL = (
    BASE_DIR
    / "dados_tratados"
    / "painel_vsr_semanal_com_maturidade.parquet"
)

ARQUIVO_PAINEL_ETAPA_20 = (
    BASE_DIR
    / "saidas"
    / "etapa_20_preparacao_modelagem"
    / "painel_modelagem_brasil_regioes.parquet"
)

ARQUIVO_CALENDARIO = (
    BASE_DIR
    / "saidas"
    / "etapa_20_preparacao_modelagem"
    / "calendario_validacao_modelagem.csv"
)

DIRETORIO_RESULTADOS = (
    BASE_DIR
    / "saidas"
    / "etapa_21_referencias_simples"
)

DIRETORIO_RESULTADOS.mkdir(parents=True, exist_ok=True)

for arquivo in [
    ARQUIVO_PAINEL_ORIGINAL,
    ARQUIVO_PAINEL_ETAPA_20,
    ARQUIVO_CALENDARIO,
]:
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")


# ------------------------------------------------------------
# Carregamento do painel e das chaves epidemiológicas
# ------------------------------------------------------------

painel_original = pd.read_parquet(ARQUIVO_PAINEL_ORIGINAL)

colunas_necessarias = [
    "nivel",
    "unidade",
    "regiao",
    "semana",
    "ano_epi",
    "semana_epi",
    "casos_vsr",
    "semana_madura_13s",
]

colunas_ausentes = [
    coluna
    for coluna in colunas_necessarias
    if coluna not in painel_original.columns
]

if colunas_ausentes:
    raise KeyError(
        f"Colunas ausentes no painel original: {colunas_ausentes}"
    )

painel = painel_original[colunas_necessarias].copy()

painel = painel.rename(
    columns={
        "semana": "data",
        "semana_madura_13s": "semana_madura",
    }
)

painel["data"] = pd.to_datetime(painel["data"], errors="coerce")
painel["casos_vsr"] = pd.to_numeric(
    painel["casos_vsr"],
    errors="coerce",
)

painel["ano_epi"] = pd.to_numeric(
    painel["ano_epi"],
    errors="coerce",
).astype("Int64")

painel["semana_epi"] = pd.to_numeric(
    painel["semana_epi"],
    errors="coerce",
).astype("Int64")

painel["nivel"] = (
    painel["nivel"].astype("string").str.strip().str.upper()
)

painel["unidade"] = (
    painel["unidade"].astype("string").str.strip().str.upper()
)

painel["regiao"] = (
    painel["regiao"].astype("string").str.strip().str.upper()
)

painel["semana_madura"] = (
    painel["semana_madura"]
    .astype("string")
    .str.strip()
    .str.upper()
    .isin(["1", "TRUE", "T", "SIM", "S"])
)

painel = painel.loc[
    painel["nivel"].isin(["BRASIL", "REGIAO"])
    & painel["semana_madura"]
].copy()

if painel["casos_vsr"].isna().any():
    raise ValueError(
        "Existem valores ausentes de casos_vsr no painel maduro."
    )

if painel[["ano_epi", "semana_epi"]].isna().any().any():
    raise ValueError(
        "Existem chaves epidemiológicas ausentes no painel maduro."
    )

if painel.duplicated(["nivel", "unidade", "data"]).any():
    raise ValueError(
        "Existem duplicidades em nível + unidade + data."
    )

painel = painel.sort_values(
    ["nivel", "unidade", "data"]
).reset_index(drop=True)


# ------------------------------------------------------------
# Confirmação de que a entrada é a mesma validada na Etapa 20
# ------------------------------------------------------------

painel_etapa_20 = pd.read_parquet(ARQUIVO_PAINEL_ETAPA_20)

painel_etapa_20["data"] = pd.to_datetime(
    painel_etapa_20["data"],
    errors="coerce",
)

painel_etapa_20["nivel"] = (
    painel_etapa_20["nivel"]
    .astype("string")
    .str.strip()
    .str.upper()
)

painel_etapa_20["unidade"] = (
    painel_etapa_20["unidade"]
    .astype("string")
    .str.strip()
    .str.upper()
)

comparacao_entrada = painel_etapa_20.merge(
    painel[
        [
            "nivel",
            "unidade",
            "data",
            "casos_vsr",
            "ano_epi",
            "semana_epi",
        ]
    ],
    on=["nivel", "unidade", "data", "casos_vsr"],
    how="outer",
    indicator=True,
)

if not (comparacao_entrada["_merge"] == "both").all():
    raise ValueError(
        "O painel da Etapa 21 não corresponde à entrada validada na Etapa 20."
    )


# ------------------------------------------------------------
# Calendário de validação temporal
# ------------------------------------------------------------

calendario = pd.read_csv(ARQUIVO_CALENDARIO)

calendario["data_origem"] = pd.to_datetime(
    calendario["data_origem"],
    errors="coerce",
)

calendario["data_alvo"] = pd.to_datetime(
    calendario["data_alvo"],
    errors="coerce",
)

calendario["nivel"] = (
    calendario["nivel"].astype("string").str.strip().str.upper()
)

calendario["unidade"] = (
    calendario["unidade"].astype("string").str.strip().str.upper()
)

if calendario[["data_origem", "data_alvo"]].isna().any().any():
    raise ValueError(
        "Existem datas inválidas no calendário de validação."
    )


# ------------------------------------------------------------
# Construção das referências de previsão
# ------------------------------------------------------------

previsoes = []
auditoria = []

for linha in calendario.itertuples(index=False):

    nivel = linha.nivel
    unidade = linha.unidade
    regiao = linha.regiao
    data_origem = linha.data_origem
    data_alvo = linha.data_alvo
    horizonte = linha.horizonte_semanas
    observado = linha.casos_observados_alvo

    serie = painel.loc[
        (painel["nivel"] == nivel)
        & (painel["unidade"] == unidade)
    ].copy()

    treino = serie.loc[
        serie["data"] <= data_origem
    ].copy()

    if treino.empty:
        raise ValueError(
            f"Não há treino disponível para {nivel} - {unidade} "
            f"na origem {data_origem.date()}."
        )

    ultimo_observado = treino.iloc[-1]

    alvo = serie.loc[
        serie["data"] == data_alvo
    ].copy()

    if len(alvo) != 1:
        raise ValueError(
            f"Não foi possível identificar o alvo de {nivel} - {unidade} "
            f"na data {data_alvo.date()}."
        )

    ano_epi_alvo = int(alvo["ano_epi"].iloc[0])
    semana_epi_alvo = int(alvo["semana_epi"].iloc[0])

    referencia_sazonal = treino.loc[
        (treino["ano_epi"] == ano_epi_alvo - 1)
        & (treino["semana_epi"] == semana_epi_alvo)
    ].copy()

    if len(referencia_sazonal) > 1:
        raise ValueError(
            f"Mais de uma referência sazonal encontrada para "
            f"{nivel} - {unidade}, alvo {data_alvo.date()}."
        )

    previsao_ingenua = float(ultimo_observado["casos_vsr"])

    if len(referencia_sazonal) == 1:
        previsao_sazonal = float(
            referencia_sazonal["casos_vsr"].iloc[0]
        )
        data_referencia_sazonal = (
            referencia_sazonal["data"].iloc[0]
        )
        referencia_sazonal_disponivel = True
    else:
        previsao_sazonal = np.nan
        data_referencia_sazonal = pd.NaT
        referencia_sazonal_disponivel = False

    previsoes.extend(
        [
            {
                "nivel": nivel,
                "unidade": unidade,
                "regiao": regiao,
                "modelo": "INGENUO",
                "data_origem": data_origem,
                "horizonte_semanas": horizonte,
                "data_alvo": data_alvo,
                "casos_observados": observado,
                "previsao": previsao_ingenua,
            },
            {
                "nivel": nivel,
                "unidade": unidade,
                "regiao": regiao,
                "modelo": "INGENUO_SAZONAL",
                "data_origem": data_origem,
                "horizonte_semanas": horizonte,
                "data_alvo": data_alvo,
                "casos_observados": observado,
                "previsao": previsao_sazonal,
            },
        ]
    )

    auditoria.append(
        {
            "nivel": nivel,
            "unidade": unidade,
            "regiao": regiao,
            "data_origem": data_origem,
            "horizonte_semanas": horizonte,
            "data_alvo": data_alvo,
            "casos_observados": observado,
            "semanas_treino_efetivas": len(treino),
            "data_ultimo_observado": ultimo_observado["data"],
            "previsao_ingenua": previsao_ingenua,
            "ano_epi_alvo": ano_epi_alvo,
            "semana_epi_alvo": semana_epi_alvo,
            "data_referencia_sazonal": data_referencia_sazonal,
            "referencia_sazonal_disponivel": referencia_sazonal_disponivel,
            "previsao_ingenua_sazonal": previsao_sazonal,
        }
    )

previsoes = pd.DataFrame(previsoes)
auditoria = pd.DataFrame(auditoria)


# ------------------------------------------------------------
# Validações das previsões geradas
# ------------------------------------------------------------

if len(previsoes) != 2 * len(calendario):
    raise ValueError(
        "A quantidade de previsões não corresponde a dois modelos "
        "por linha do calendário."
    )

if previsoes.duplicated(
    [
        "nivel",
        "unidade",
        "modelo",
        "data_origem",
        "horizonte_semanas",
    ]
).any():
    raise ValueError(
        "Foram encontradas previsões duplicadas."
    )

if previsoes.loc[
    previsoes["modelo"] == "INGENUO",
    "previsao"
].isna().any():
    raise ValueError(
        "A referência ingênua possui previsões ausentes."
    )

resumo_disponibilidade = (
    previsoes
    .assign(previsao_disponivel=previsoes["previsao"].notna())
    .groupby(["modelo", "horizonte_semanas"])
    .agg(
        previsoes_esperadas=("previsao", "size"),
        previsoes_disponiveis=("previsao_disponivel", "sum"),
    )
    .reset_index()
)

resumo_disponibilidade["percentual_disponivel"] = (
    100
    * resumo_disponibilidade["previsoes_disponiveis"]
    / resumo_disponibilidade["previsoes_esperadas"]
)


# ------------------------------------------------------------
# Gráfico ilustrativo: último corte temporal do Brasil
# ------------------------------------------------------------

ultimo_corte_brasil = (
    calendario.loc[
        (calendario["nivel"] == "BRASIL")
        & (calendario["unidade"] == "BRASIL"),
        "data_origem",
    ]
    .max()
)

serie_brasil = painel.loc[
    (painel["nivel"] == "BRASIL")
    & (painel["unidade"] == "BRASIL")
].copy()

historico_exemplo = serie_brasil.loc[
    (serie_brasil["data"] >= ultimo_corte_brasil - pd.Timedelta(weeks=104))
    & (serie_brasil["data"] <= ultimo_corte_brasil)
].copy()

exemplo = previsoes.loc[
    (previsoes["nivel"] == "BRASIL")
    & (previsoes["unidade"] == "BRASIL")
    & (previsoes["data_origem"] == ultimo_corte_brasil)
].copy()

previsoes_exemplo = exemplo.pivot(
    index="data_alvo",
    columns="modelo",
    values="previsao",
).reset_index()

observados_exemplo = (
    exemplo[
        ["data_alvo", "casos_observados"]
    ]
    .drop_duplicates()
    .sort_values("data_alvo")
)

fig, ax = plt.subplots(figsize=(13, 5.5))

ax.plot(
    historico_exemplo["data"],
    historico_exemplo["casos_vsr"],
    color="#1264A3",
    linewidth=2,
    label="Histórico disponível",
)

ax.axvline(
    ultimo_corte_brasil,
    color="#E85F5C",
    linestyle="--",
    linewidth=1.5,
    label="Origem da previsão",
)

ax.plot(
    observados_exemplo["data_alvo"],
    observados_exemplo["casos_observados"],
    "o-",
    color="#111827",
    linewidth=2,
    label="Observado depois da origem",
)

ax.plot(
    previsoes_exemplo["data_alvo"],
    previsoes_exemplo["INGENUO"],
    "s--",
    color="#F6A041",
    linewidth=2,
    label="Ingênuo",
)

ax.plot(
    previsoes_exemplo["data_alvo"],
    previsoes_exemplo["INGENUO_SAZONAL"],
    "D--",
    color="#32B4C8",
    linewidth=2,
    label="Ingênuo sazonal",
)

ax.set_title(
    "Brasil — exemplo de previsões de referência no último corte temporal",
    loc="left",
    fontweight="bold",
)

ax.set_xlabel("Semana de início dos sintomas")
ax.set_ylabel("Casos de VSR")
ax.grid(alpha=0.25)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()

fig.savefig(
    DIRETORIO_RESULTADOS
    / "exemplo_previsao_referencias_brasil.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# Salvamento dos resultados
# ------------------------------------------------------------

previsoes.to_csv(
    DIRETORIO_RESULTADOS
    / "previsoes_referencias_simples.csv",
    index=False,
)

auditoria.to_csv(
    DIRETORIO_RESULTADOS
    / "auditoria_referencias_simples.csv",
    index=False,
)

resumo_disponibilidade.to_csv(
    DIRETORIO_RESULTADOS
    / "resumo_disponibilidade_referencias.csv",
    index=False,
)

print("\nEtapa 21 concluída com sucesso.")
print(f"Previsões geradas: {len(previsoes):,}")
print(f"Linhas de auditoria: {len(auditoria):,}")
print(f"\nSaídas salvas em:\n{DIRETORIO_RESULTADOS}")

display(resumo_disponibilidade)

## Etapa 22 — Métricas de erro das referências

Nesta etapa, calcularemos os erros das previsões ingênua e ingênua sazonal para cada série e horizonte de previsão.

O objetivo é comparar as referências de forma justa ao longo de todos os cortes temporais. Serão utilizados MAE, que mede o erro médio em número de casos; RMSE, que dá maior peso aos erros grandes; e MASE, que coloca o erro em uma escala sazonal comparável entre Brasil e regiões.

In [ ]:
# ETAPA 22 — Cálculo das métricas de erro das referências

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path("/content/tratamento_vsr")

ARQUIVO_PAINEL_ORIGINAL = (
    BASE_DIR
    / "dados_tratados"
    / "painel_vsr_semanal_com_maturidade.parquet"
)

ARQUIVO_PREVISOES = (
    BASE_DIR
    / "saidas"
    / "etapa_21_referencias_simples"
    / "previsoes_referencias_simples.csv"
)

DIRETORIO_RESULTADOS = (
    BASE_DIR
    / "saidas"
    / "etapa_22_metricas_referencias"
)

DIRETORIO_RESULTADOS.mkdir(parents=True, exist_ok=True)

for arquivo in [ARQUIVO_PAINEL_ORIGINAL, ARQUIVO_PREVISOES]:
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")


# ------------------------------------------------------------
# Leitura das previsões produzidas na Etapa 21
# ------------------------------------------------------------

previsoes = pd.read_csv(ARQUIVO_PREVISOES)

for coluna in ["data_origem", "data_alvo"]:
    previsoes[coluna] = pd.to_datetime(
        previsoes[coluna],
        errors="coerce",
    )

previsoes["casos_observados"] = pd.to_numeric(
    previsoes["casos_observados"],
    errors="coerce",
)

previsoes["previsao"] = pd.to_numeric(
    previsoes["previsao"],
    errors="coerce",
)

if previsoes[
    ["data_origem", "data_alvo", "casos_observados", "previsao"]
].isna().any().any():
    raise ValueError(
        "Existem valores ausentes nas previsões da Etapa 21."
    )

if len(previsoes) != 2784:
    raise ValueError(
        f"Foram encontradas {len(previsoes):,} previsões. "
        "O esperado é 2.784 previsões."
    )


# ------------------------------------------------------------
# Leitura da base semanal para cálculo da escala do MASE
# ------------------------------------------------------------

painel_original = pd.read_parquet(ARQUIVO_PAINEL_ORIGINAL)

colunas_necessarias = [
    "nivel",
    "unidade",
    "regiao",
    "semana",
    "ano_epi",
    "semana_epi",
    "casos_vsr",
    "semana_madura_13s",
]

colunas_ausentes = [
    coluna
    for coluna in colunas_necessarias
    if coluna not in painel_original.columns
]

if colunas_ausentes:
    raise KeyError(
        f"Colunas ausentes no painel original: {colunas_ausentes}"
    )

painel = painel_original[colunas_necessarias].copy()

painel = painel.rename(
    columns={
        "semana": "data",
        "semana_madura_13s": "semana_madura",
    }
)

painel["data"] = pd.to_datetime(
    painel["data"],
    errors="coerce",
)

painel["casos_vsr"] = pd.to_numeric(
    painel["casos_vsr"],
    errors="coerce",
)

painel["ano_epi"] = pd.to_numeric(
    painel["ano_epi"],
    errors="coerce",
).astype("Int64")

painel["semana_epi"] = pd.to_numeric(
    painel["semana_epi"],
    errors="coerce",
).astype("Int64")

painel["nivel"] = (
    painel["nivel"].astype("string").str.strip().str.upper()
)

painel["unidade"] = (
    painel["unidade"].astype("string").str.strip().str.upper()
)

painel["regiao"] = (
    painel["regiao"].astype("string").str.strip().str.upper()
)

if pd.api.types.is_bool_dtype(painel["semana_madura"]):
    painel["semana_madura"] = painel["semana_madura"].fillna(False)
else:
    painel["semana_madura"] = (
        painel["semana_madura"]
        .astype("string")
        .str.strip()
        .str.upper()
        .isin(["1", "TRUE", "T", "SIM", "S"])
    )

painel = painel.loc[
    painel["nivel"].isin(["BRASIL", "REGIAO"])
    & painel["semana_madura"]
].copy()

if painel["casos_vsr"].isna().any():
    raise ValueError(
        "Existem casos ausentes no painel maduro usado para calcular MASE."
    )

if painel[["ano_epi", "semana_epi"]].isna().any().any():
    raise ValueError(
        "Existem chaves epidemiológicas ausentes no painel maduro."
    )


# ------------------------------------------------------------
# Escala sazonal do MASE para cada origem temporal
# ------------------------------------------------------------

def calcular_escala_mase_sazonal(treino):
    treino = treino.sort_values("data").copy()

    if treino.duplicated(["ano_epi", "semana_epi"]).any():
        raise ValueError(
            "Há duplicidade de chave epidemiológica na série de treino."
        )

    valores_por_semana = {
        (int(linha.ano_epi), int(linha.semana_epi)): float(linha.casos_vsr)
        for linha in treino.itertuples(index=False)
    }

    erros_sazonais = []

    for linha in treino.itertuples(index=False):
        chave_anterior = (
            int(linha.ano_epi) - 1,
            int(linha.semana_epi),
        )

        if chave_anterior in valores_por_semana:
            erro = abs(
                float(linha.casos_vsr)
                - valores_por_semana[chave_anterior]
            )
            erros_sazonais.append(erro)

    if len(erros_sazonais) == 0:
        return np.nan, 0

    return float(np.mean(erros_sazonais)), len(erros_sazonais)


origens = (
    previsoes[
        ["nivel", "unidade", "regiao", "data_origem"]
    ]
    .drop_duplicates()
    .sort_values(["nivel", "unidade", "data_origem"])
)

escalas_mase = []

for linha in origens.itertuples(index=False):

    treino = painel.loc[
        (painel["nivel"] == linha.nivel)
        & (painel["unidade"] == linha.unidade)
        & (painel["data"] <= linha.data_origem)
    ].copy()

    escala, comparacoes_sazonais = calcular_escala_mase_sazonal(
        treino
    )

    escalas_mase.append(
        {
            "nivel": linha.nivel,
            "unidade": linha.unidade,
            "regiao": linha.regiao,
            "data_origem": linha.data_origem,
            "semanas_treino": len(treino),
            "comparacoes_sazonais_treino": comparacoes_sazonais,
            "escala_mase_sazonal": escala,
        }
    )

escalas_mase = pd.DataFrame(escalas_mase)

if escalas_mase["escala_mase_sazonal"].isna().any():
    raise ValueError(
        "Não foi possível calcular a escala sazonal do MASE "
        "para alguma origem temporal."
    )

if (escalas_mase["escala_mase_sazonal"] <= 0).any():
    raise ValueError(
        "Foi encontrada escala MASE menor ou igual a zero."
    )


# ------------------------------------------------------------
# Cálculo dos erros individuais
# ------------------------------------------------------------

erros = previsoes.merge(
    escalas_mase[
        [
            "nivel",
            "unidade",
            "data_origem",
            "semanas_treino",
            "comparacoes_sazonais_treino",
            "escala_mase_sazonal",
        ]
    ],
    on=["nivel", "unidade", "data_origem"],
    how="left",
    validate="many_to_one",
)

if erros["escala_mase_sazonal"].isna().any():
    raise ValueError(
        "Existem previsões sem escala MASE correspondente."
    )

erros["erro"] = (
    erros["previsao"]
    - erros["casos_observados"]
)

erros["erro_absoluto"] = erros["erro"].abs()

erros["erro_quadratico"] = erros["erro"] ** 2

erros["mase_individual"] = (
    erros["erro_absoluto"]
    / erros["escala_mase_sazonal"]
)


# ------------------------------------------------------------
# Métricas por série, modelo e horizonte
# ------------------------------------------------------------

metricas_por_serie = (
    erros
    .groupby(
        [
            "nivel",
            "unidade",
            "regiao",
            "modelo",
            "horizonte_semanas",
        ],
        dropna=False,
    )
    .agg(
        avaliacoes=("erro_absoluto", "size"),
        mae=("erro_absoluto", "mean"),
        rmse=(
            "erro_quadratico",
            lambda valores: float(np.sqrt(np.mean(valores))),
        ),
        mase=("mase_individual", "mean"),
    )
    .reset_index()
)

metricas_agregadas = (
    erros
    .groupby(
        ["modelo", "horizonte_semanas"],
        dropna=False,
    )
    .agg(
        avaliacoes=("erro_absoluto", "size"),
        mae=("erro_absoluto", "mean"),
        rmse=(
            "erro_quadratico",
            lambda valores: float(np.sqrt(np.mean(valores))),
        ),
        mase=("mase_individual", "mean"),
    )
    .reset_index()
)

metricas_agregadas = metricas_agregadas.sort_values(
    ["horizonte_semanas", "modelo"]
).reset_index(drop=True)

ranking_referencias = metricas_agregadas.copy()

ranking_referencias["ranking_mae"] = (
    ranking_referencias
    .groupby("horizonte_semanas")["mae"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

ranking_referencias["ranking_rmse"] = (
    ranking_referencias
    .groupby("horizonte_semanas")["rmse"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

ranking_referencias["ranking_mase"] = (
    ranking_referencias
    .groupby("horizonte_semanas")["mase"]
    .rank(method="dense", ascending=True)
    .astype(int)
)


# ------------------------------------------------------------
# Gráfico comparativo das referências
# ------------------------------------------------------------

metricas_grafico = metricas_agregadas.copy()

modelos = ["INGENUO", "INGENUO_SAZONAL"]
horizontes = [1, 4, 8, 13]

rotulos_modelos = {
    "INGENUO": "Ingênuo",
    "INGENUO_SAZONAL": "Ingênuo sazonal",
}

cores_modelos = {
    "INGENUO": "#F6A041",
    "INGENUO_SAZONAL": "#32B4C8",
}

fig, eixos = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(17, 4.8),
)

metricas_plot = [
    ("mae", "MAE\n(casos)"),
    ("rmse", "RMSE\n(casos)"),
    ("mase", "MASE\n(escala sazonal)"),
]

largura = 0.34
posicoes = np.arange(len(horizontes))

for eixo, (coluna, titulo) in zip(eixos, metricas_plot):

    for indice, modelo in enumerate(modelos):

        valores = (
            metricas_grafico
            .loc[
                metricas_grafico["modelo"] == modelo,
                ["horizonte_semanas", coluna],
            ]
            .set_index("horizonte_semanas")
            .reindex(horizontes)[coluna]
        )

        deslocamento = (
            posicoes
            + (indice - 0.5) * largura
        )

        eixo.bar(
            deslocamento,
            valores,
            width=largura,
            label=rotulos_modelos[modelo],
            color=cores_modelos[modelo],
        )

    eixo.set_title(titulo, fontweight="bold")
    eixo.set_xticks(posicoes)
    eixo.set_xticklabels(
        [f"{h} sem." for h in horizontes]
    )
    eixo.grid(axis="y", alpha=0.25)

eixos[0].set_ylabel("Erro")
eixos[2].legend(
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
)

fig.suptitle(
    "Comparação das referências simples por horizonte de previsão",
    fontsize=15,
    fontweight="bold",
    y=1.03,
)

fig.tight_layout()

fig.savefig(
    DIRETORIO_RESULTADOS
    / "comparacao_metricas_referencias.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ------------------------------------------------------------
# Salvamento das saídas
# ------------------------------------------------------------

escalas_mase.to_csv(
    DIRETORIO_RESULTADOS
    / "escalas_mase_por_origem.csv",
    index=False,
)

erros.to_csv(
    DIRETORIO_RESULTADOS
    / "erros_individuais_referencias.csv",
    index=False,
)

metricas_por_serie.to_csv(
    DIRETORIO_RESULTADOS
    / "metricas_por_serie_modelo_horizonte.csv",
    index=False,
)

metricas_agregadas.to_csv(
    DIRETORIO_RESULTADOS
    / "metricas_agregadas_referencias.csv",
    index=False,
)

ranking_referencias.to_csv(
    DIRETORIO_RESULTADOS
    / "ranking_referencias_por_horizonte.csv",
    index=False,
)

print("\nEtapa 22 concluída com sucesso.")
print(f"Escalas MASE calculadas: {len(escalas_mase):,}")
print(f"Erros individuais calculados: {len(erros):,}")
print(f"Métricas por série: {len(metricas_por_serie):,}")
print(f"Métricas agregadas: {len(metricas_agregadas):,}")
print(f"\nSaídas salvas em:\n{DIRETORIO_RESULTADOS}")

display(metricas_agregadas)
display(ranking_referencias)

## Etapa 23 — Modelos ETS com sazonalidade anual

Nesta etapa, ajustaremos modelos de suavização exponencial para Brasil e regiões, utilizando nível, tendência e sazonalidade anual.

O objetivo é verificar se modelos ETS conseguem combinar o comportamento recente da série com o padrão sazonal observado, superando as referências ingênua e ingênua sazonal nos mesmos cortes temporais.

In [ ]:
# ETAPA 23 — Modelos ETS com sazonalidade anual e auditoria de convergência

import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.holtwinters import ExponentialSmoothing

BASE_DIR = Path("/content/tratamento_vsr")

ARQUIVO_PAINEL = (
    BASE_DIR
    / "saidas"
    / "etapa_20_preparacao_modelagem"
    / "painel_modelagem_brasil_regioes.parquet"
)

ARQUIVO_CALENDARIO = (
    BASE_DIR
    / "saidas"
    / "etapa_20_preparacao_modelagem"
    / "calendario_validacao_modelagem.csv"
)

DIRETORIO_RESULTADOS = (
    BASE_DIR
    / "saidas"
    / "etapa_23_modelos_ets"
)

DIRETORIO_RESULTADOS.mkdir(parents=True, exist_ok=True)

for arquivo in [ARQUIVO_PAINEL, ARQUIVO_CALENDARIO]:
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

painel = pd.read_parquet(ARQUIVO_PAINEL)
calendario = pd.read_csv(ARQUIVO_CALENDARIO)

painel["data"] = pd.to_datetime(painel["data"], errors="coerce")
calendario["data_origem"] = pd.to_datetime(
    calendario["data_origem"],
    errors="coerce",
)
calendario["data_alvo"] = pd.to_datetime(
    calendario["data_alvo"],
    errors="coerce",
)

for base in [painel, calendario]:
    base["nivel"] = base["nivel"].astype("string").str.strip().str.upper()
    base["unidade"] = base["unidade"].astype("string").str.strip().str.upper()

painel["casos_vsr"] = pd.to_numeric(
    painel["casos_vsr"],
    errors="coerce",
)

if painel[["data", "casos_vsr"]].isna().any().any():
    raise ValueError("Existem datas ou casos ausentes no painel de modelagem.")

if calendario[
    ["data_origem", "data_alvo", "casos_observados_alvo"]
].isna().any().any():
    raise ValueError("Existem valores ausentes no calendário de validação.")

PERIODO_SAZONAL = 52
HORIZONTES = sorted(calendario["horizonte_semanas"].unique())
HORIZONTE_MAXIMO = int(max(HORIZONTES))

CONFIGURACOES_ETS = {
    "ETS_SAZONAL_ADITIVO": {
        "trend": None,
        "damped_trend": False,
        "seasonal": "add",
    },
    "ETS_TENDENCIA_ADITIVA": {
        "trend": "add",
        "damped_trend": False,
        "seasonal": "add",
    },
    "ETS_TENDENCIA_AMORTECIDA": {
        "trend": "add",
        "damped_trend": True,
        "seasonal": "add",
    },
}

origens = (
    calendario[
        [
            "nivel",
            "unidade",
            "regiao",
            "data_origem",
            "semanas_treino",
        ]
    ]
    .drop_duplicates()
    .sort_values(["nivel", "unidade", "data_origem"])
    .reset_index(drop=True)
)

regularidade = []

for (nivel, unidade), serie in painel.groupby(["nivel", "unidade"]):
    diferencas = (
        serie.sort_values("data")["data"]
        .diff()
        .dt.days
        .dropna()
    )

    regularidade.append(
        {
            "nivel": nivel,
            "unidade": unidade,
            "semanas": len(serie),
            "primeira_semana": serie["data"].min(),
            "ultima_semana": serie["data"].max(),
            "intervalo_minimo_dias": diferencas.min(),
            "intervalo_maximo_dias": diferencas.max(),
            "serie_semanal_regular": diferencas.eq(7).all(),
        }
    )

regularidade = pd.DataFrame(regularidade)

if not regularidade["serie_semanal_regular"].all():
    display(regularidade.loc[~regularidade["serie_semanal_regular"]])
    raise ValueError("Foram encontradas lacunas nas séries semanais.")

def ajustar_ets_com_auditoria(valores_treino, parametros):
    tentativas = [
        {
            "numero_tentativa": 1,
            "otimizador": "padrão",
            "argumentos_fit": {
                "optimized": True,
                "remove_bias": False,
                "use_brute": True,
            },
        },
        {
            "numero_tentativa": 2,
            "otimizador": "L-BFGS-B",
            "argumentos_fit": {
                "optimized": True,
                "remove_bias": False,
                "use_brute": False,
                "method": "L-BFGS-B",
                "minimize_kwargs": {
                    "options": {
                        "maxiter": 2000,
                    }
                },
            },
        },
    ]

    historico = []

    for tentativa in tentativas:
        try:
            modelo = ExponentialSmoothing(
                valores_treino,
                trend=parametros["trend"],
                damped_trend=parametros["damped_trend"],
                seasonal=parametros["seasonal"],
                seasonal_periods=PERIODO_SAZONAL,
                initialization_method="estimated",
            )

            with warnings.catch_warnings(record=True) as avisos:
                warnings.simplefilter("always")
                ajuste = modelo.fit(**tentativa["argumentos_fit"])

            avisos_convergencia = [
                aviso
                for aviso in avisos
                if issubclass(aviso.category, ConvergenceWarning)
            ]

            retorno_otimizador = getattr(ajuste, "mle_retvals", {})
            sucesso_otimizador = np.nan

            if isinstance(retorno_otimizador, dict):
                sucesso_otimizador = retorno_otimizador.get(
                    "success",
                    np.nan,
                )

            convergiu = (
                len(avisos_convergencia) == 0
                and sucesso_otimizador is not False
            )

            mensagens = " | ".join(
                sorted(
                    {
                        str(aviso.message)
                        for aviso in avisos
                    }
                )
            )

            historico.append(
                {
                    "numero_tentativa": tentativa["numero_tentativa"],
                    "otimizador": tentativa["otimizador"],
                    "status_tentativa": (
                        "CONVERGIU"
                        if convergiu
                        else "NAO_CONVERGIU"
                    ),
                    "quantidade_avisos": len(avisos),
                    "quantidade_avisos_convergencia": len(
                        avisos_convergencia
                    ),
                    "sucesso_reportado_otimizador": sucesso_otimizador,
                    "mensagens_avisos": mensagens,
                    "mensagem_erro": None,
                }
            )

            if convergiu:
                return ajuste, historico

        except Exception as erro:
            historico.append(
                {
                    "numero_tentativa": tentativa["numero_tentativa"],
                    "otimizador": tentativa["otimizador"],
                    "status_tentativa": "ERRO",
                    "quantidade_avisos": 0,
                    "quantidade_avisos_convergencia": 0,
                    "sucesso_reportado_otimizador": np.nan,
                    "mensagens_avisos": None,
                    "mensagem_erro": str(erro),
                }
            )

    return None, historico

previsoes_ets = []
auditoria_ets = []
auditoria_tentativas = []

inicio_execucao = time.time()
total_origens = len(origens)

for indice_origem, origem in origens.iterrows():
    nivel = origem["nivel"]
    unidade = origem["unidade"]
    regiao = origem["regiao"]
    data_origem = origem["data_origem"]
    semanas_treino_esperadas = int(origem["semanas_treino"])

    if (
        indice_origem == 0
        or (indice_origem + 1) % 20 == 0
        or (indice_origem + 1) == total_origens
    ):
        minutos = (time.time() - inicio_execucao) / 60
        print(
            f"Processando origem {indice_origem + 1:,} de "
            f"{total_origens:,} "
            f"({minutos:.1f} minutos decorridos)."
        )

    serie = painel.loc[
        (painel["nivel"] == nivel)
        & (painel["unidade"] == unidade)
    ].sort_values("data")

    treino = serie.loc[serie["data"] <= data_origem].copy()

    if len(treino) != semanas_treino_esperadas:
        raise ValueError(
            f"Treino inconsistente em {nivel} - {unidade}, "
            f"origem {data_origem.date()}."
        )

    alvos_origem = calendario.loc[
        (calendario["nivel"] == nivel)
        & (calendario["unidade"] == unidade)
        & (calendario["data_origem"] == data_origem)
    ].sort_values("horizonte_semanas")

    valores_treino = treino["casos_vsr"].astype(float).to_numpy()

    for nome_modelo, parametros in CONFIGURACOES_ETS.items():
        ajuste, tentativas = ajustar_ets_com_auditoria(
            valores_treino,
            parametros,
        )

        for tentativa in tentativas:
            auditoria_tentativas.append(
                {
                    "nivel": nivel,
                    "unidade": unidade,
                    "regiao": regiao,
                    "modelo": nome_modelo,
                    "data_origem": data_origem,
                    "semanas_treino": len(treino),
                    **tentativa,
                }
            )

        if ajuste is None:
            auditoria_ets.append(
                {
                    "nivel": nivel,
                    "unidade": unidade,
                    "regiao": regiao,
                    "modelo": nome_modelo,
                    "data_origem": data_origem,
                    "semanas_treino": len(treino),
                    "status_final": "NAO_CONVERGIU_APOS_REAJUSTE",
                    "tentativa_escolhida": np.nan,
                    "aic": np.nan,
                    "bic": np.nan,
                    "sse": np.nan,
                    "previsao_gerada": False,
                    "mensagem_final": tentativas[-1]["mensagem_erro"],
                }
            )
            continue

        try:
            previsao_completa = np.asarray(
                ajuste.forecast(HORIZONTE_MAXIMO),
                dtype=float,
            )

            if len(previsao_completa) != HORIZONTE_MAXIMO:
                raise ValueError("Quantidade inesperada de horizontes.")

            if not np.isfinite(previsao_completa).all():
                raise ValueError("O ajuste gerou previsão não finita.")

            tentativa_escolhida = [
                item["numero_tentativa"]
                for item in tentativas
                if item["status_tentativa"] == "CONVERGIU"
            ][0]

            auditoria_ets.append(
                {
                    "nivel": nivel,
                    "unidade": unidade,
                    "regiao": regiao,
                    "modelo": nome_modelo,
                    "data_origem": data_origem,
                    "semanas_treino": len(treino),
                    "status_final": "CONVERGIU",
                    "tentativa_escolhida": tentativa_escolhida,
                    "aic": getattr(ajuste, "aic", np.nan),
                    "bic": getattr(ajuste, "bic", np.nan),
                    "sse": getattr(ajuste, "sse", np.nan),
                    "previsao_gerada": True,
                    "mensagem_final": None,
                }
            )

            for alvo in alvos_origem.itertuples(index=False):
                horizonte = int(alvo.horizonte_semanas)
                previsao_bruta = float(
                    previsao_completa[horizonte - 1]
                )

                previsoes_ets.append(
                    {
                        "nivel": nivel,
                        "unidade": unidade,
                        "regiao": regiao,
                        "modelo": nome_modelo,
                        "data_origem": data_origem,
                        "horizonte_semanas": horizonte,
                        "data_alvo": alvo.data_alvo,
                        "casos_observados": alvo.casos_observados_alvo,
                        "previsao_bruta": previsao_bruta,
                        "previsao": max(0.0, previsao_bruta),
                        "previsao_ajustada_nao_negativa": (
                            previsao_bruta < 0
                        ),
                    }
                )

        except Exception as erro:
            auditoria_ets.append(
                {
                    "nivel": nivel,
                    "unidade": unidade,
                    "regiao": regiao,
                    "modelo": nome_modelo,
                    "data_origem": data_origem,
                    "semanas_treino": len(treino),
                    "status_final": "ERRO_NA_PREVISAO",
                    "tentativa_escolhida": np.nan,
                    "aic": np.nan,
                    "bic": np.nan,
                    "sse": np.nan,
                    "previsao_gerada": False,
                    "mensagem_final": str(erro),
                }
            )

previsoes_ets = pd.DataFrame(previsoes_ets)
auditoria_ets = pd.DataFrame(auditoria_ets)
auditoria_tentativas = pd.DataFrame(auditoria_tentativas)

if previsoes_ets.empty:
    raise RuntimeError(
        "Nenhum ajuste ETS convergiu. "
        "Consulte a auditoria de tentativas."
    )

resumo_disponibilidade_ets = (
    previsoes_ets
    .groupby(["modelo", "horizonte_semanas"])
    .agg(
        previsoes_disponiveis=("previsao", "size"),
        previsoes_ajustadas_nao_negativas=(
            "previsao_ajustada_nao_negativa",
            "sum",
        ),
    )
    .reindex(
        pd.MultiIndex.from_product(
            [
                list(CONFIGURACOES_ETS.keys()),
                HORIZONTES,
            ],
            names=["modelo", "horizonte_semanas"],
        ),
        fill_value=0,
    )
    .reset_index()
)

resumo_disponibilidade_ets["previsoes_esperadas"] = len(origens)

resumo_disponibilidade_ets["percentual_disponivel"] = (
    100
    * resumo_disponibilidade_ets["previsoes_disponiveis"]
    / resumo_disponibilidade_ets["previsoes_esperadas"]
)

resumo_convergencia_ets = (
    auditoria_ets
    .groupby(["modelo", "status_final"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

ultimo_corte_brasil = calendario.loc[
    (calendario["nivel"] == "BRASIL")
    & (calendario["unidade"] == "BRASIL"),
    "data_origem",
].max()

serie_brasil = painel.loc[
    (painel["nivel"] == "BRASIL")
    & (painel["unidade"] == "BRASIL")
].copy()

historico_exemplo = serie_brasil.loc[
    (
        serie_brasil["data"]
        >= ultimo_corte_brasil - pd.Timedelta(weeks=104)
    )
    & (serie_brasil["data"] <= ultimo_corte_brasil)
]

observados_exemplo = (
    calendario.loc[
        (calendario["nivel"] == "BRASIL")
        & (calendario["unidade"] == "BRASIL")
        & (calendario["data_origem"] == ultimo_corte_brasil),
        ["data_alvo", "casos_observados_alvo"],
    ]
    .drop_duplicates()
    .sort_values("data_alvo")
)

previsoes_exemplo = previsoes_ets.loc[
    (previsoes_ets["nivel"] == "BRASIL")
    & (previsoes_ets["unidade"] == "BRASIL")
    & (previsoes_ets["data_origem"] == ultimo_corte_brasil)
].copy()

cores_ets = {
    "ETS_SAZONAL_ADITIVO": "#32B4C8",
    "ETS_TENDENCIA_ADITIVA": "#7C3AED",
    "ETS_TENDENCIA_AMORTECIDA": "#F6A041",
}

rotulos_ets = {
    "ETS_SAZONAL_ADITIVO": "ETS sazonal aditivo",
    "ETS_TENDENCIA_ADITIVA": "ETS tendência aditiva",
    "ETS_TENDENCIA_AMORTECIDA": "ETS tendência amortecida",
}

fig, ax = plt.subplots(figsize=(13, 5.5))

ax.plot(
    historico_exemplo["data"],
    historico_exemplo["casos_vsr"],
    color="#1264A3",
    linewidth=2,
    label="Histórico disponível",
)

ax.axvline(
    ultimo_corte_brasil,
    color="#E85F5C",
    linestyle="--",
    linewidth=1.5,
    label="Origem da previsão",
)

ax.plot(
    observados_exemplo["data_alvo"],
    observados_exemplo["casos_observados_alvo"],
    "o-",
    color="#111827",
    linewidth=2,
    label="Observado depois da origem",
)

for modelo, grupo in previsoes_exemplo.groupby("modelo"):
    grupo = grupo.sort_values("data_alvo")

    ax.plot(
        grupo["data_alvo"],
        grupo["previsao"],
        "D--",
        color=cores_ets[modelo],
        linewidth=2,
        label=rotulos_ets[modelo],
    )

ax.set_title(
    "Brasil — previsões ETS no último corte temporal",
    loc="left",
    fontweight="bold",
)
ax.set_xlabel("Semana de início dos sintomas")
ax.set_ylabel("Casos de VSR")
ax.grid(alpha=0.25)
ax.legend(frameon=False, ncol=2)

fig.tight_layout()

fig.savefig(
    DIRETORIO_RESULTADOS / "exemplo_previsoes_ets_brasil.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()

regularidade.to_csv(
    DIRETORIO_RESULTADOS / "controle_regularidade_series_ets.csv",
    index=False,
)

pd.DataFrame(
    [
        {
            "modelo": nome,
            "tendencia": parametros["trend"],
            "tendencia_amortecida": parametros["damped_trend"],
            "sazonalidade": parametros["seasonal"],
            "periodo_sazonal": PERIODO_SAZONAL,
        }
        for nome, parametros in CONFIGURACOES_ETS.items()
    ]
).to_csv(
    DIRETORIO_RESULTADOS / "configuracoes_ets.csv",
    index=False,
)

previsoes_ets.to_csv(
    DIRETORIO_RESULTADOS / "previsoes_ets.csv",
    index=False,
)

auditoria_ets.to_csv(
    DIRETORIO_RESULTADOS / "auditoria_ets.csv",
    index=False,
)

auditoria_tentativas.to_csv(
    DIRETORIO_RESULTADOS / "auditoria_tentativas_ets.csv",
    index=False,
)

resumo_disponibilidade_ets.to_csv(
    DIRETORIO_RESULTADOS / "resumo_disponibilidade_ets.csv",
    index=False,
)

resumo_convergencia_ets.to_csv(
    DIRETORIO_RESULTADOS / "resumo_convergencia_ets.csv",
    index=False,
)

print("\nEtapa 23 concluída.")
print(f"Origens temporais: {len(origens):,}")
print(f"Ajustes ETS previstos: {len(origens) * len(CONFIGURACOES_ETS):,}")
print(f"Previsões ETS disponíveis: {len(previsoes_ets):,}")
print(f"\nSaídas salvas em:\n{DIRETORIO_RESULTADOS}")

display(resumo_convergencia_ets)
display(resumo_disponibilidade_ets)

## Etapa 23B — ETS com inicialização sazonal heurística

A etapa 23 mostrou que a regularidade semanal dos dados está adequada, mas que parte dos ajustes ETS com sazonalidade anual não convergiu de forma confiável. Esse problema foi mais frequente quando incluímos tendência e tendência amortecida.

Nesta etapa, manteremos exatamente as mesmas seis séries, os mesmos cortes temporais, os mesmos horizontes de previsão e as mesmas três estruturas ETS. A única mudança será a inicialização heurística dos componentes sazonais.

O objetivo é verificar se a instabilidade anterior foi causada principalmente pela estimação simultânea de muitos estados iniciais — incluindo 52 componentes sazonais — em cortes com histórico ainda relativamente curto.

Ao final, cada ajuste será classificado como convergente, não convergente ou com erro. Um modelo somente será aprovado para a comparação principal de erros se produzir previsões para todos os 348 cortes temporais.

Assim, esta etapa não altera o problema nem escolhe um modelo antecipadamente. Ela testa, de forma controlada, se uma formulação ETS mais parcimoniosa é numericamente estável para as séries de internações por VSR.

In [ ]:
# ETAPA 23B — ETS com inicialização sazonal heurística

import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.holtwinters import ExponentialSmoothing

BASE_DIR = Path("/content/tratamento_vsr")

ARQUIVO_PAINEL = (
    BASE_DIR
    / "saidas"
    / "etapa_20_preparacao_modelagem"
    / "painel_modelagem_brasil_regioes.parquet"
)

ARQUIVO_CALENDARIO = (
    BASE_DIR
    / "saidas"
    / "etapa_20_preparacao_modelagem"
    / "calendario_validacao_modelagem.csv"
)

DIRETORIO_RESULTADOS = (
    BASE_DIR
    / "saidas"
    / "etapa_23b_ets_inicializacao_heuristica"
)

DIRETORIO_RESULTADOS.mkdir(parents=True, exist_ok=True)

for arquivo in [ARQUIVO_PAINEL, ARQUIVO_CALENDARIO]:
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

painel = pd.read_parquet(ARQUIVO_PAINEL)
calendario = pd.read_csv(ARQUIVO_CALENDARIO)

painel["data"] = pd.to_datetime(painel["data"], errors="coerce")
calendario["data_origem"] = pd.to_datetime(
    calendario["data_origem"],
    errors="coerce",
)
calendario["data_alvo"] = pd.to_datetime(
    calendario["data_alvo"],
    errors="coerce",
)

for base in [painel, calendario]:
    base["nivel"] = base["nivel"].astype("string").str.strip().str.upper()
    base["unidade"] = base["unidade"].astype("string").str.strip().str.upper()

painel["casos_vsr"] = pd.to_numeric(
    painel["casos_vsr"],
    errors="coerce",
)

if painel[["data", "casos_vsr"]].isna().any().any():
    raise ValueError("Existem datas ou casos ausentes no painel.")

if calendario[
    ["data_origem", "data_alvo", "casos_observados_alvo"]
].isna().any().any():
    raise ValueError("Existem valores ausentes no calendário.")

PERIODO_SAZONAL = 52
HORIZONTES = sorted(calendario["horizonte_semanas"].unique())
HORIZONTE_MAXIMO = int(max(HORIZONTES))

CONFIGURACOES_ETS = {
    "ETS_SAZONAL_ADITIVO_HEURISTICO": {
        "trend": None,
        "damped_trend": False,
        "seasonal": "add",
    },
    "ETS_TENDENCIA_ADITIVA_HEURISTICO": {
        "trend": "add",
        "damped_trend": False,
        "seasonal": "add",
    },
    "ETS_TENDENCIA_AMORTECIDA_HEURISTICO": {
        "trend": "add",
        "damped_trend": True,
        "seasonal": "add",
    },
}

origens = (
    calendario[
        [
            "nivel",
            "unidade",
            "regiao",
            "data_origem",
            "semanas_treino",
        ]
    ]
    .drop_duplicates()
    .sort_values(["nivel", "unidade", "data_origem"])
    .reset_index(drop=True)
)

regularidade = []

for (nivel, unidade), serie in painel.groupby(["nivel", "unidade"]):
    diferencas = (
        serie.sort_values("data")["data"]
        .diff()
        .dt.days
        .dropna()
    )

    regularidade.append(
        {
            "nivel": nivel,
            "unidade": unidade,
            "semanas": len(serie),
            "primeira_semana": serie["data"].min(),
            "ultima_semana": serie["data"].max(),
            "intervalo_minimo_dias": diferencas.min(),
            "intervalo_maximo_dias": diferencas.max(),
            "serie_semanal_regular": diferencas.eq(7).all(),
        }
    )

regularidade = pd.DataFrame(regularidade)

if not regularidade["serie_semanal_regular"].all():
    display(regularidade.loc[~regularidade["serie_semanal_regular"]])
    raise ValueError("Foram encontradas lacunas nas séries semanais.")

def ajustar_ets_heuristico(valores_treino, parametros):
    tentativas = [
        {
            "numero_tentativa": 1,
            "otimizador": "padrão",
            "argumentos_fit": {
                "optimized": True,
                "remove_bias": False,
                "use_brute": True,
            },
        },
        {
            "numero_tentativa": 2,
            "otimizador": "L-BFGS-B",
            "argumentos_fit": {
                "optimized": True,
                "remove_bias": False,
                "use_brute": False,
                "method": "L-BFGS-B",
                "minimize_kwargs": {
                    "options": {
                        "maxiter": 2000,
                    }
                },
            },
        },
    ]

    historico = []

    for tentativa in tentativas:
        try:
            modelo = ExponentialSmoothing(
                valores_treino,
                trend=parametros["trend"],
                damped_trend=parametros["damped_trend"],
                seasonal=parametros["seasonal"],
                seasonal_periods=PERIODO_SAZONAL,
                initialization_method="heuristic",
            )

            with warnings.catch_warnings(record=True) as avisos:
                warnings.simplefilter("always")
                ajuste = modelo.fit(**tentativa["argumentos_fit"])

            avisos_convergencia = [
                aviso
                for aviso in avisos
                if issubclass(aviso.category, ConvergenceWarning)
            ]

            retorno_otimizador = getattr(ajuste, "mle_retvals", {})
            sucesso_otimizador = np.nan

            if isinstance(retorno_otimizador, dict):
                sucesso_otimizador = retorno_otimizador.get(
                    "success",
                    np.nan,
                )

            convergiu = (
                len(avisos_convergencia) == 0
                and sucesso_otimizador is not False
            )

            mensagens = " | ".join(
                sorted({str(aviso.message) for aviso in avisos})
            )

            historico.append(
                {
                    "numero_tentativa": tentativa["numero_tentativa"],
                    "otimizador": tentativa["otimizador"],
                    "status_tentativa": (
                        "CONVERGIU"
                        if convergiu
                        else "NAO_CONVERGIU"
                    ),
                    "quantidade_avisos": len(avisos),
                    "quantidade_avisos_convergencia": len(
                        avisos_convergencia
                    ),
                    "sucesso_reportado_otimizador": sucesso_otimizador,
                    "mensagens_avisos": mensagens,
                    "mensagem_erro": None,
                }
            )

            if convergiu:
                return ajuste, historico

        except Exception as erro:
            historico.append(
                {
                    "numero_tentativa": tentativa["numero_tentativa"],
                    "otimizador": tentativa["otimizador"],
                    "status_tentativa": "ERRO",
                    "quantidade_avisos": 0,
                    "quantidade_avisos_convergencia": 0,
                    "sucesso_reportado_otimizador": np.nan,
                    "mensagens_avisos": None,
                    "mensagem_erro": str(erro),
                }
            )

    return None, historico

linhas_previsoes = []
linhas_auditoria = []
linhas_tentativas = []

inicio_execucao = time.time()
total_origens = len(origens)

for indice_origem, origem in origens.iterrows():
    nivel = origem["nivel"]
    unidade = origem["unidade"]
    regiao = origem["regiao"]
    data_origem = origem["data_origem"]
    semanas_treino_esperadas = int(origem["semanas_treino"])

    if (
        indice_origem == 0
        or (indice_origem + 1) % 20 == 0
        or (indice_origem + 1) == total_origens
    ):
        minutos = (time.time() - inicio_execucao) / 60

        print(
            f"Processando origem {indice_origem + 1:,} de "
            f"{total_origens:,} "
            f"({minutos:.1f} minutos decorridos)."
        )

    serie = painel.loc[
        (painel["nivel"] == nivel)
        & (painel["unidade"] == unidade)
    ].sort_values("data")

    treino = serie.loc[serie["data"] <= data_origem].copy()

    if len(treino) != semanas_treino_esperadas:
        raise ValueError(
            f"Treino inconsistente em {nivel} - {unidade}, "
            f"origem {data_origem.date()}."
        )

    alvos_origem = calendario.loc[
        (calendario["nivel"] == nivel)
        & (calendario["unidade"] == unidade)
        & (calendario["data_origem"] == data_origem)
    ].sort_values("horizonte_semanas")

    valores_treino = treino["casos_vsr"].astype(float).to_numpy()

    for nome_modelo, parametros in CONFIGURACOES_ETS.items():
        ajuste, tentativas = ajustar_ets_heuristico(
            valores_treino,
            parametros,
        )

        for tentativa in tentativas:
            linhas_tentativas.append(
                {
                    "nivel": nivel,
                    "unidade": unidade,
                    "regiao": regiao,
                    "modelo": nome_modelo,
                    "data_origem": data_origem,
                    "semanas_treino": len(treino),
                    **tentativa,
                }
            )

        if ajuste is None:
            linhas_auditoria.append(
                {
                    "nivel": nivel,
                    "unidade": unidade,
                    "regiao": regiao,
                    "modelo": nome_modelo,
                    "data_origem": data_origem,
                    "semanas_treino": len(treino),
                    "status_final": "NAO_CONVERGIU_APOS_REAJUSTE",
                    "tentativa_escolhida": np.nan,
                    "aic": np.nan,
                    "bic": np.nan,
                    "sse": np.nan,
                    "previsao_gerada": False,
                }
            )
            continue

        try:
            previsao_completa = np.asarray(
                ajuste.forecast(HORIZONTE_MAXIMO),
                dtype=float,
            )

            if len(previsao_completa) != HORIZONTE_MAXIMO:
                raise ValueError("Horizontes retornados incorretamente.")

            if not np.isfinite(previsao_completa).all():
                raise ValueError("Foram geradas previsões não finitas.")

            tentativa_escolhida = [
                item["numero_tentativa"]
                for item in tentativas
                if item["status_tentativa"] == "CONVERGIU"
            ][0]

            linhas_auditoria.append(
                {
                    "nivel": nivel,
                    "unidade": unidade,
                    "regiao": regiao,
                    "modelo": nome_modelo,
                    "data_origem": data_origem,
                    "semanas_treino": len(treino),
                    "status_final": "CONVERGIU",
                    "tentativa_escolhida": tentativa_escolhida,
                    "aic": getattr(ajuste, "aic", np.nan),
                    "bic": getattr(ajuste, "bic", np.nan),
                    "sse": getattr(ajuste, "sse", np.nan),
                    "previsao_gerada": True,
                }
            )

            for alvo in alvos_origem.itertuples(index=False):
                horizonte = int(alvo.horizonte_semanas)
                previsao_bruta = float(
                    previsao_completa[horizonte - 1]
                )

                linhas_previsoes.append(
                    {
                        "nivel": nivel,
                        "unidade": unidade,
                        "regiao": regiao,
                        "modelo": nome_modelo,
                        "data_origem": data_origem,
                        "horizonte_semanas": horizonte,
                        "data_alvo": alvo.data_alvo,
                        "casos_observados": alvo.casos_observados_alvo,
                        "previsao_bruta": previsao_bruta,
                        "previsao": max(0.0, previsao_bruta),
                        "previsao_ajustada_nao_negativa": (
                            previsao_bruta < 0
                        ),
                    }
                )

        except Exception:
            linhas_auditoria.append(
                {
                    "nivel": nivel,
                    "unidade": unidade,
                    "regiao": regiao,
                    "modelo": nome_modelo,
                    "data_origem": data_origem,
                    "semanas_treino": len(treino),
                    "status_final": "ERRO_NA_PREVISAO",
                    "tentativa_escolhida": np.nan,
                    "aic": np.nan,
                    "bic": np.nan,
                    "sse": np.nan,
                    "previsao_gerada": False,
                }
            )

previsoes_ets = pd.DataFrame(linhas_previsoes)
auditoria_ets = pd.DataFrame(linhas_auditoria)
auditoria_tentativas = pd.DataFrame(linhas_tentativas)

resumo_convergencia = (
    auditoria_ets
    .groupby(["modelo", "status_final"])
    .size()
    .unstack(fill_value=0)
    .reindex(list(CONFIGURACOES_ETS.keys()), fill_value=0)
    .rename_axis("modelo")
    .reset_index()
)

resumo_convergencia["ajustes_esperados"] = len(origens)

resumo_convergencia["modelo_aprovado_para_metricas"] = (
    resumo_convergencia.get("CONVERGIU", 0)
    == resumo_convergencia["ajustes_esperados"]
)

resumo_disponibilidade = (
    previsoes_ets
    .groupby(["modelo", "horizonte_semanas"])
    .agg(
        previsoes_disponiveis=("previsao", "size"),
        previsoes_ajustadas_nao_negativas=(
            "previsao_ajustada_nao_negativa",
            "sum",
        ),
    )
    .reindex(
        pd.MultiIndex.from_product(
            [
                list(CONFIGURACOES_ETS.keys()),
                HORIZONTES,
            ],
            names=["modelo", "horizonte_semanas"],
        ),
        fill_value=0,
    )
    .reset_index()
)

resumo_disponibilidade["previsoes_esperadas"] = len(origens)

resumo_disponibilidade["percentual_disponivel"] = (
    100
    * resumo_disponibilidade["previsoes_disponiveis"]
    / resumo_disponibilidade["previsoes_esperadas"]
)

ultimo_corte_brasil = calendario.loc[
    (calendario["nivel"] == "BRASIL")
    & (calendario["unidade"] == "BRASIL"),
    "data_origem",
].max()

serie_brasil = painel.loc[
    (painel["nivel"] == "BRASIL")
    & (painel["unidade"] == "BRASIL")
].copy()

historico_exemplo = serie_brasil.loc[
    (
        serie_brasil["data"]
        >= ultimo_corte_brasil - pd.Timedelta(weeks=104)
    )
    & (serie_brasil["data"] <= ultimo_corte_brasil)
]

observados_exemplo = (
    calendario.loc[
        (calendario["nivel"] == "BRASIL")
        & (calendario["unidade"] == "BRASIL")
        & (calendario["data_origem"] == ultimo_corte_brasil),
        ["data_alvo", "casos_observados_alvo"],
    ]
    .drop_duplicates()
    .sort_values("data_alvo")
)

previsoes_exemplo = previsoes_ets.loc[
    (previsoes_ets["nivel"] == "BRASIL")
    & (previsoes_ets["unidade"] == "BRASIL")
    & (previsoes_ets["data_origem"] == ultimo_corte_brasil)
].copy()

cores = {
    "ETS_SAZONAL_ADITIVO_HEURISTICO": "#32B4C8",
    "ETS_TENDENCIA_ADITIVA_HEURISTICO": "#7C3AED",
    "ETS_TENDENCIA_AMORTECIDA_HEURISTICO": "#F6A041",
}

rotulos = {
    "ETS_SAZONAL_ADITIVO_HEURISTICO": "ETS sazonal aditivo",
    "ETS_TENDENCIA_ADITIVA_HEURISTICO": "ETS tendência aditiva",
    "ETS_TENDENCIA_AMORTECIDA_HEURISTICO": (
        "ETS tendência amortecida"
    ),
}

fig, ax = plt.subplots(figsize=(13, 5.5))

ax.plot(
    historico_exemplo["data"],
    historico_exemplo["casos_vsr"],
    color="#1264A3",
    linewidth=2,
    label="Histórico disponível",
)

ax.axvline(
    ultimo_corte_brasil,
    color="#E85F5C",
    linestyle="--",
    linewidth=1.5,
    label="Origem da previsão",
)

ax.plot(
    observados_exemplo["data_alvo"],
    observados_exemplo["casos_observados_alvo"],
    "o-",
    color="#111827",
    linewidth=2,
    label="Observado depois da origem",
)

for modelo, grupo in previsoes_exemplo.groupby("modelo"):
    grupo = grupo.sort_values("data_alvo")

    ax.plot(
        grupo["data_alvo"],
        grupo["previsao"],
        "D--",
        color=cores[modelo],
        linewidth=2,
        label=rotulos[modelo],
    )

ax.set_title(
    "Brasil — ETS com inicialização heurística no último corte",
    loc="left",
    fontweight="bold",
)
ax.set_xlabel("Semana de início dos sintomas")
ax.set_ylabel("Casos de VSR")
ax.grid(alpha=0.25)
ax.legend(frameon=False, ncol=2)

fig.tight_layout()

fig.savefig(
    DIRETORIO_RESULTADOS
    / "exemplo_previsoes_ets_heuristico_brasil.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()

regularidade.to_csv(
    DIRETORIO_RESULTADOS
    / "controle_regularidade_series_ets_heuristico.csv",
    index=False,
)

pd.DataFrame(
    [
        {
            "modelo": nome,
            "tendencia": parametros["trend"],
            "tendencia_amortecida": parametros["damped_trend"],
            "sazonalidade": parametros["seasonal"],
            "periodo_sazonal": PERIODO_SAZONAL,
            "inicializacao": "heuristic",
        }
        for nome, parametros in CONFIGURACOES_ETS.items()
    ]
).to_csv(
    DIRETORIO_RESULTADOS / "configuracoes_ets_heuristico.csv",
    index=False,
)

previsoes_ets.to_csv(
    DIRETORIO_RESULTADOS / "previsoes_ets_heuristico.csv",
    index=False,
)

auditoria_ets.to_csv(
    DIRETORIO_RESULTADOS / "auditoria_ets_heuristico.csv",
    index=False,
)

auditoria_tentativas.to_csv(
    DIRETORIO_RESULTADOS
    / "auditoria_tentativas_ets_heuristico.csv",
    index=False,
)

resumo_convergencia.to_csv(
    DIRETORIO_RESULTADOS
    / "resumo_convergencia_ets_heuristico.csv",
    index=False,
)

resumo_disponibilidade.to_csv(
    DIRETORIO_RESULTADOS
    / "resumo_disponibilidade_ets_heuristico.csv",
    index=False,
)

print("\nEtapa 23B concluída.")
print(f"Origens temporais: {len(origens):,}")
print(
    "Ajustes ETS previstos: "
    f"{len(origens) * len(CONFIGURACOES_ETS):,}"
)
print(f"Previsões disponíveis: {len(previsoes_ets):,}")
print(f"\nSaídas salvas em:\n{DIRETORIO_RESULTADOS}")

display(resumo_convergencia)
display(resumo_disponibilidade)

## Etapa 24 — Comparação entre baselines e modelos ETS

Nesta etapa, compararemos as previsões das duas referências simples e dos três modelos ETS com inicialização heurística.

Todos os modelos serão avaliados nos mesmos cortes temporais, horizontes de 1, 4, 8 e 13 semanas, séries e valores observados. Assim, diferenças de desempenho poderão ser atribuídas aos modelos, e não a mudanças na base ou no período avaliado.

A avaliação usará MAE, RMSE e MASE. As previsões ETS utilizadas serão as previsões operacionais não negativas; a quantidade de previsões corrigidas para zero será registrada separadamente como diagnóstico de plausibilidade.

O objetivo não é escolher o menor número isolado. Um modelo será considerado mais útil se apresentar erro menor ou competitivo, comportamento estável entre horizontes e séries, e previsões coerentes com contagens de internações.

Ao final, teremos uma evidência objetiva para decidir quais estruturas ETS seguem como candidatas e quais devem ser descartadas antes da comparação com modelos SARIMA.

In [ ]:
# ETAPA 24 — Comparação de baselines e modelos ETS

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = Path("/content/tratamento_vsr")
DIRETORIO_SAIDAS = BASE_DIR / "saidas"
DIRETORIO_RESULTADOS = (
    BASE_DIR
    / "saidas"
    / "etapa_24_comparacao_baselines_ets"
)

DIRETORIO_RESULTADOS.mkdir(parents=True, exist_ok=True)

def localizar_arquivo(nome_arquivo):
    encontrados = list(DIRETORIO_SAIDAS.rglob(nome_arquivo))

    if len(encontrados) == 0:
        raise FileNotFoundError(
            f"Não foi encontrado o arquivo: {nome_arquivo}"
        )

    if len(encontrados) > 1:
        raise ValueError(
            f"Foi encontrada mais de uma versão de: {nome_arquivo}\n"
            f"{encontrados}"
        )

    return encontrados[0]

ARQUIVO_BASELINES = localizar_arquivo(
    "previsoes_referencias_simples.csv"
)

ARQUIVO_ESCALAS_MASE = localizar_arquivo(
    "escalas_mase_por_origem.csv"
)

ARQUIVO_ETS = localizar_arquivo(
    "previsoes_ets_heuristico.csv"
)

baselines = pd.read_csv(ARQUIVO_BASELINES)
escalas_mase = pd.read_csv(ARQUIVO_ESCALAS_MASE)
ets = pd.read_csv(ARQUIVO_ETS)

for base in [baselines, escalas_mase, ets]:
    for coluna in ["nivel", "unidade"]:
        base[coluna] = (
            base[coluna]
            .astype("string")
            .str.strip()
            .str.upper()
        )

for base in [baselines, escalas_mase, ets]:
    if "data_origem" in base.columns:
        base["data_origem"] = pd.to_datetime(
            base["data_origem"],
            errors="coerce",
        )

if "data_alvo" in baselines.columns:
    baselines["data_alvo"] = pd.to_datetime(
        baselines["data_alvo"],
        errors="coerce",
    )

if "data_alvo" in ets.columns:
    ets["data_alvo"] = pd.to_datetime(
        ets["data_alvo"],
        errors="coerce",
    )

for base in [baselines, ets]:
    base["casos_observados"] = pd.to_numeric(
        base["casos_observados"],
        errors="coerce",
    )

    base["previsao"] = pd.to_numeric(
        base["previsao"],
        errors="coerce",
    )

    base["horizonte_semanas"] = pd.to_numeric(
        base["horizonte_semanas"],
        errors="coerce",
    ).astype("Int64")

escalas_mase["escala_mase_sazonal"] = pd.to_numeric(
    escalas_mase["escala_mase_sazonal"],
    errors="coerce",
)

if ets["previsao_ajustada_nao_negativa"].dtype == object:
    ets["previsao_ajustada_nao_negativa"] = (
        ets["previsao_ajustada_nao_negativa"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("true")
    )

baselines["familia_modelo"] = "BASELINE"
ets["familia_modelo"] = "ETS"

colunas_previsao = [
    "nivel",
    "unidade",
    "regiao",
    "modelo",
    "familia_modelo",
    "data_origem",
    "horizonte_semanas",
    "data_alvo",
    "casos_observados",
    "previsao",
]

baselines = baselines[colunas_previsao].copy()

ets = ets.assign(
    previsao_ajustada_nao_negativa=(
        ets["previsao_ajustada_nao_negativa"]
    )
)[
    colunas_previsao
    + ["previsao_ajustada_nao_negativa"]
].copy()

baselines["previsao_ajustada_nao_negativa"] = False

previsoes = pd.concat(
    [baselines, ets],
    ignore_index=True,
)

chaves_previsao = [
    "nivel",
    "unidade",
    "modelo",
    "data_origem",
    "horizonte_semanas",
]

if previsoes.duplicated(chaves_previsao).any():
    raise ValueError("Foram encontradas previsões duplicadas.")

if previsoes[
    ["casos_observados", "previsao"]
].isna().any().any():
    raise ValueError(
        "Foram encontrados valores ausentes nas previsões."
    )

controle_modelos = (
    previsoes
    .groupby(["familia_modelo", "modelo"])
    .agg(
        previsoes_geradas=("previsao", "size"),
        origens_temporais=("data_origem", "nunique"),
        series=("unidade", "nunique"),
        horizontes=("horizonte_semanas", "nunique"),
        correcoes_nao_negativas=(
            "previsao_ajustada_nao_negativa",
            "sum",
        ),
    )
    .reset_index()
)

controle_modelos["percentual_correcoes_nao_negativas"] = (
    100
    * controle_modelos["correcoes_nao_negativas"]
    / controle_modelos["previsoes_geradas"]
)

quantidade_esperada_por_modelo = (
    previsoes[
        ["nivel", "unidade", "data_origem", "horizonte_semanas"]
    ]
    .drop_duplicates()
    .shape[0]
)

if not (
    controle_modelos["previsoes_geradas"]
    == quantidade_esperada_por_modelo
).all():
    display(controle_modelos)

    raise ValueError(
        "Nem todos os modelos possuem a mesma quantidade de previsões."
    )

chaves_escala = [
    "nivel",
    "unidade",
    "data_origem",
]

previsoes = previsoes.merge(
    escalas_mase[
        chaves_escala
        + [
            "semanas_treino",
            "comparacoes_sazonais_treino",
            "escala_mase_sazonal",
        ]
    ],
    on=chaves_escala,
    how="left",
    validate="many_to_one",
)

if previsoes["escala_mase_sazonal"].isna().any():
    raise ValueError(
        "Existem previsões sem escala MASE correspondente."
    )

if (previsoes["escala_mase_sazonal"] <= 0).any():
    raise ValueError(
        "Existem escalas MASE nulas ou negativas."
    )

previsoes["erro"] = (
    previsoes["previsao"]
    - previsoes["casos_observados"]
)

previsoes["erro_absoluto"] = previsoes["erro"].abs()
previsoes["erro_quadratico"] = previsoes["erro"] ** 2

previsoes["erro_absoluto_escalado"] = (
    previsoes["erro_absoluto"]
    / previsoes["escala_mase_sazonal"]
)

def calcular_metricas(grupo):
    return pd.Series(
        {
            "avaliacoes": len(grupo),
            "mae": grupo["erro_absoluto"].mean(),
            "rmse": np.sqrt(grupo["erro_quadratico"].mean()),
            "mase": grupo["erro_absoluto_escalado"].mean(),
            "viés_medio": grupo["erro"].mean(),
        }
    )

metricas_por_serie = (
    previsoes
    .groupby(
        [
            "familia_modelo",
            "modelo",
            "nivel",
            "unidade",
            "horizonte_semanas",
        ]
    )
    .apply(calcular_metricas)
    .reset_index()
)

metricas_agregadas = (
    previsoes
    .groupby(
        [
            "familia_modelo",
            "modelo",
            "horizonte_semanas",
        ]
    )
    .apply(calcular_metricas)
    .reset_index()
)

ranking_modelos = metricas_agregadas.copy()

ranking_modelos["ranking_mase"] = (
    ranking_modelos
    .groupby("horizonte_semanas")["mase"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

ranking_modelos["ranking_mae"] = (
    ranking_modelos
    .groupby("horizonte_semanas")["mae"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

ranking_modelos["ranking_rmse"] = (
    ranking_modelos
    .groupby("horizonte_semanas")["rmse"]
    .rank(method="dense", ascending=True)
    .astype(int)
)

ranking_modelos = ranking_modelos.sort_values(
    [
        "horizonte_semanas",
        "ranking_mase",
        "ranking_mae",
        "ranking_rmse",
    ]
)

correcoes_ets = (
    previsoes.loc[previsoes["familia_modelo"] == "ETS"]
    .groupby(["modelo", "horizonte_semanas"])
    .agg(
        previsoes=("previsao", "size"),
        previsoes_corrigidas=(
            "previsao_ajustada_nao_negativa",
            "sum",
        ),
    )
    .reset_index()
)

correcoes_ets["percentual_corrigido"] = (
    100
    * correcoes_ets["previsoes_corrigidas"]
    / correcoes_ets["previsoes"]
)

ordem_modelos = [
    "INGENUO",
    "INGENUO_SAZONAL",
    "ETS_SAZONAL_ADITIVO_HEURISTICO",
    "ETS_TENDENCIA_ADITIVA_HEURISTICO",
    "ETS_TENDENCIA_AMORTECIDA_HEURISTICO",
]

rotulos_modelos = {
    "INGENUO": "Ingênuo",
    "INGENUO_SAZONAL": "Ingênuo sazonal",
    "ETS_SAZONAL_ADITIVO_HEURISTICO": "ETS sazonal",
    "ETS_TENDENCIA_ADITIVA_HEURISTICO": "ETS tendência",
    "ETS_TENDENCIA_AMORTECIDA_HEURISTICO": (
        "ETS tendência amortecida"
    ),
}

cores_modelos = {
    "INGENUO": "#64748B",
    "INGENUO_SAZONAL": "#0EA5E9",
    "ETS_SAZONAL_ADITIVO_HEURISTICO": "#14B8A6",
    "ETS_TENDENCIA_ADITIVA_HEURISTICO": "#8B5CF6",
    "ETS_TENDENCIA_AMORTECIDA_HEURISTICO": "#F59E0B",
}

fig, eixos = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(15, 5.5),
)

for modelo in ordem_modelos:
    dados = metricas_agregadas.loc[
        metricas_agregadas["modelo"] == modelo
    ].sort_values("horizonte_semanas")

    eixos[0].plot(
        dados["horizonte_semanas"],
        dados["mase"],
        marker="o",
        linewidth=2,
        color=cores_modelos[modelo],
        label=rotulos_modelos[modelo],
    )

    eixos[1].plot(
        dados["horizonte_semanas"],
        dados["mae"],
        marker="o",
        linewidth=2,
        color=cores_modelos[modelo],
        label=rotulos_modelos[modelo],
    )

eixos[0].set_title(
    "Erro absoluto escalado por horizonte",
    loc="left",
    fontweight="bold",
)
eixos[0].set_xlabel("Horizonte de previsão (semanas)")
eixos[0].set_ylabel("MASE")
eixos[0].set_xticks([1, 4, 8, 13])
eixos[0].grid(alpha=0.25)

eixos[1].set_title(
    "Erro absoluto médio por horizonte",
    loc="left",
    fontweight="bold",
)
eixos[1].set_xlabel("Horizonte de previsão (semanas)")
eixos[1].set_ylabel("MAE em número de casos")
eixos[1].set_xticks([1, 4, 8, 13])
eixos[1].grid(alpha=0.25)

handles, labels = eixos[1].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=3,
    frameon=False,
)

fig.tight_layout(rect=[0, 0.10, 1, 1])

fig.savefig(
    DIRETORIO_RESULTADOS
    / "comparacao_baselines_ets_metricas.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()

controle_modelos.to_csv(
    DIRETORIO_RESULTADOS
    / "controle_modelos_comparados.csv",
    index=False,
)

previsoes.to_csv(
    DIRETORIO_RESULTADOS
    / "erros_individuais_baselines_ets.csv",
    index=False,
)

metricas_por_serie.to_csv(
    DIRETORIO_RESULTADOS
    / "metricas_por_serie_modelo_horizonte.csv",
    index=False,
)

metricas_agregadas.to_csv(
    DIRETORIO_RESULTADOS
    / "metricas_agregadas_baselines_ets.csv",
    index=False,
)

ranking_modelos.to_csv(
    DIRETORIO_RESULTADOS
    / "ranking_modelos_baselines_ets.csv",
    index=False,
)

correcoes_ets.to_csv(
    DIRETORIO_RESULTADOS
    / "diagnostico_correcoes_nao_negativas_ets.csv",
    index=False,
)

print("\nEtapa 24 concluída.")
print(
    f"Modelos comparados: "
    f"{controle_modelos['modelo'].nunique()}"
)
print(
    f"Previsões por modelo: "
    f"{quantidade_esperada_por_modelo:,}"
)
print(f"\nSaídas salvas em:\n{DIRETORIO_RESULTADOS}")

display(controle_modelos)
display(
    ranking_modelos[
        [
            "horizonte_semanas",
            "modelo",
            "avaliacoes",
            "mae",
            "rmse",
            "mase",
            "viés_medio",
            "ranking_mase",
        ]
    ]
)